# Google ADK with Couchbase via Model Context Protocol (MCP) - A Tutorial

This notebook demonstrates how to build an intelligent AI agent using [Google's Agent Development Kit (ADK)](https://google.github.io/adk-docs/) that can interact with a Couchbase database. The key to this interaction is the **Model Context Protocol (MCP)**, which allows the AI agent to seamlessly connect to and query Couchbase using natural language.

## What You'll Build

By the end of this tutorial, you will have an AI agent that can:
- Connect to a Couchbase cluster and explore its structure
- Discover buckets, scopes, collections, and document schemas
- Run SQL++ queries through natural language conversations
- Analyze query performance and get index recommendations

## What is the Model Context Protocol (MCP)?

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) is an open standard that standardizes how AI assistants connect to external data sources and tools. Think of it as a **universal adapter** between AI models and the systems they need to interact with.

**How MCP Works:**

* **MCP Server** — A lightweight program that exposes tools from an external system (in our case, Couchbase). The [Couchbase MCP Server](https://github.com/Couchbase-Ecosystem/mcp-server-couchbase) provides tools for querying, exploring schemas, checking cluster health, and more.
* **MCP Client** — An application that connects to the MCP server and makes its tools available to an AI model. Google ADK's `McpToolset` acts as the MCP client in this tutorial.
* **Communication** — The client and server communicate over **stdio** (standard input/output) when running locally. The MCP server runs as a subprocess, and ADK handles all the communication automatically.

This architecture means your agent doesn't need the Couchbase SDK installed — the MCP server handles all database communication behind the scenes.

## What is Google ADK?

[Google's Agent Development Kit (ADK)](https://google.github.io/adk-docs/) is a framework for building AI agents powered by Gemini models. It provides:

* **LlmAgent** — The core agent class that takes user queries, decides which tools to call, and generates natural language responses.
* **McpToolset** — A toolset that connects to any MCP server, automatically discovering all available tools.
* **Runner** — Manages the agent's execution loop, handling the back-and-forth between the LLM and tools.
* **Built-in Web UI** — A local web interface for testing agents interactively.

# Before You Start

## Get Credentials for Google Gemini

You'll need a Google Gemini API key to power the AI agent. Get one for free from [Google AI Studio](https://aistudio.google.com/).

## Create and Deploy Your Free Tier Operational Cluster on Capella

To get started with Couchbase Capella, create an account and use it to deploy a forever free tier operational cluster. This account provides you with an environment where you can explore and learn about Capella with no time constraint.

To learn more, please follow the [instructions](https://docs.couchbase.com/cloud/get-started/create-account.html).

### Couchbase Capella Configuration

When running Couchbase using [Capella](https://cloud.couchbase.com/sign-in), the following prerequisites need to be met:

* Create the [database credentials](https://docs.couchbase.com/cloud/clusters/manage-database-users.html) to access the required bucket (Read and Write) used in the application.
* [Allow access](https://docs.couchbase.com/cloud/clusters/allow-ip-address.html) to the Cluster from the IP on which the application is running.
* Your Capella free-tier account includes a `travel-sample` bucket, with sample documents used for booking and travel purposes. You can find more information [here](https://docs.couchbase.com/cloud/get-started/run-first-queries.html).

## Setup Instructions

Before running this notebook, ensure you have the following prerequisites met:

* **Set Environment Variables:** This notebook loads the Gemini API key and Couchbase credentials from a `.env` file. Include the following:

    ```
    GOOGLE_GENAI_API_KEY=your_gemini_api_key_here
    CB_CONNECTION_STRING=your_couchbase_connection_string
    CB_USERNAME=your_couchbase_username
    CB_PASSWORD=your_couchbase_password
    ```

    We have already included a `.env.sample` file. Rename it to `.env` and fill in the values.

* **Setup uv:** `uv` is a modern and fast Python package manager. We use it to run the MCP server without manual installation. Install uv from [here](https://docs.astral.sh/uv/getting-started/installation/#installing-uv).

* **Python Libraries:** Install the necessary libraries by running the code cell below.

In [1]:
%pip install -q 'google-adk>=1.18.0' 'python-dotenv>=1.0.0'

Note: you may need to restart the kernel to use updated packages.


## Importing Libraries and Loading Environment Variables

Let's import everything we need:

* **`dotenv`** — Loads API keys and database credentials from a `.env` file so we don't hardcode secrets.
* **`LlmAgent`** — The main agent class from Google ADK. It takes a model, instructions, and tools, and handles the reasoning loop.
* **`Runner`** — Executes the agent, managing the conversation flow and tool calls.
* **`InMemorySessionService`** — Stores conversation history in memory (for this tutorial; production apps would use persistent storage).
* **`McpToolset` and `StdioConnectionParams`** — Connect to the Couchbase MCP Server over stdio.
* **`StdioServerParameters`** — Configures how the MCP server subprocess is launched.
* **`types`** — Google GenAI types for constructing messages.

In [2]:
import os
import logging
import warnings

from dotenv import load_dotenv
from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from google.genai import types
from mcp import StdioServerParameters

# Suppress noisy logs from MCP subprocess
logging.getLogger("couchbase").setLevel(logging.ERROR)
logging.getLogger("mcp").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# Load environment variables from .env file
load_dotenv()

True

## Loading Couchbase Credentials

We read the Couchbase connection details from environment variables. These will be passed to the MCP server so it can connect to your cluster.

**Important:** Environment variables loaded via `dotenv` in this notebook do NOT automatically propagate to the MCP server subprocess. That's why we read them here and explicitly pass them in the `env` dict when configuring the MCP connection later.

In [3]:
CB_CONNECTION_STRING = os.getenv("CB_CONNECTION_STRING")
CB_USERNAME = os.getenv("CB_USERNAME")
CB_PASSWORD = os.getenv("CB_PASSWORD")

# Verify credentials are loaded
if not all([CB_CONNECTION_STRING, CB_USERNAME, CB_PASSWORD]):
    raise ValueError(
        "Missing environment variables. Please create a .env file with "
        "GOOGLE_GENAI_API_KEY, CB_CONNECTION_STRING, CB_USERNAME, CB_PASSWORD"
    )

print(f"Connection String: {CB_CONNECTION_STRING}")
print(f"Username: {CB_USERNAME}")
print("Password: ****")

Connection String: couchbase://127.0.0.1
Username: kaustav
Password: ****


## Defining the System Prompt

The system prompt is a crucial piece of instruction given to the LLM that powers our agent. It tells the model:

1. **What it is** — A Couchbase database assistant
2. **What data exists** — The `travel-sample` bucket with `inventory` scope containing airlines, airports, hotels, landmarks, and routes
3. **How to query** — SQL++ syntax rules specific to Couchbase (e.g., use backticks for field names, UNNEST for arrays)
4. **What NOT to do** — Never answer from its own knowledge; always query the database

A well-crafted system prompt is the difference between an agent that generates correct SQL++ queries and one that hallucinates results. The more specific you are about the data structure and query rules, the better the agent performs.

In [4]:
system_prompt = """You are a Couchbase database assistant connected to
the `travel-sample` bucket (scope: `inventory`). The connection is already
established — do NOT call `test_cluster_connection` (it has known issues);
just answer using the schema and query tools.

Always use the MCP tools to query the database — never answer from memory.
Do not ask the user clarifying questions about the schema; the cheatsheet
below tells you everything you need.

Available collections in `inventory`: airline, airport, hotel, landmark, route.

Schema cheatsheet:
- `hotel`: name, city, country, address, price, reviews[*].ratings.{Overall, Cleanliness, Service}
- `route`: airline, sourceairport, destinationairport, distance, schedule[*].flight
- `landmark`: name, city, country, activity, content, price

SQL++ rules:
- Use only the collection name in the FROM clause (e.g. `FROM hotel h`).
  Do NOT prefix with the bucket or scope.
- For aggregations over array fields, flatten with UNNEST first.

Common query patterns to use directly:

"Top hotels by rating" — sum the Overall review rating per hotel:
    SELECT h.name, SUM(r.ratings.Overall) AS total
    FROM hotel h UNNEST h.reviews r
    GROUP BY h.name
    ORDER BY total DESC
    LIMIT 5

"Flights from X to Y":
    SELECT r.airline, r.sourceairport, r.destinationairport, r.distance
    FROM route r
    WHERE r.sourceairport = "X" AND r.destinationairport = "Y"

`hotel.price` and `landmark.price` are free-form strings (e.g. "$54-$104",
"From £50") or null — never filter them numerically. For budget questions:
    SELECT h.name, h.city, h.price
    FROM hotel h
    WHERE h.city = "X" AND h.price IS NOT NULL
Then read each price string yourself and pick the ones that fit.
"""

## Creating the Agent

Now we create the AI agent. This is where everything comes together:

* **`model="gemini-2.5-flash"`** — The Gemini model that powers the agent's reasoning. `gemini-2.5-flash` handles the multi-step reasoning and conversational follow-ups in this tutorial reliably and is the recommended default. The free tier allows 250 requests/day, which is plenty for tutorial use. If you hit that limit while iterating, you can swap in `gemini-2.5-flash-lite` (1000 requests/day, but may struggle with the more open-ended questions) or, for maximum reasoning quality, `gemini-2.5-pro`. All three are drop-in replacements.
* **`instruction=system_prompt`** — The system prompt we defined above, which tells the agent how to behave.
* **`tools=[McpToolset(...)]`** — The connection to the Couchbase MCP Server. Let's break down this configuration:
  * `command="uvx"` and `args=["couchbase-mcp-server"]` — Launches the MCP server using `uv`. This automatically downloads and runs the server without manual installation.
  * `env={...}` — Passes Couchbase credentials to the MCP server subprocess. This is required because the subprocess doesn't inherit environment variables from `dotenv`.
  * `CB_MCP_READ_ONLY_MODE="true"` — Restricts the server to read-only operations, preventing accidental data modifications. Set to `"false"` if you need write access.
  * `timeout=60` — Allows up to 60 seconds for the MCP server to start.

In [5]:
root_agent = LlmAgent(
    model="gemini-2.5-flash",
    name="couchbase_agent",
    description=(
        "An agent that interacts with Couchbase databases using the "
        "Couchbase MCP server."
    ),
    instruction=system_prompt,
    tools=[
        McpToolset(
            connection_params=StdioConnectionParams(
                server_params=StdioServerParameters(
                    command="uvx",
                    args=["couchbase-mcp-server"],
                    env={
                        "CB_CONNECTION_STRING": CB_CONNECTION_STRING,
                        "CB_USERNAME": CB_USERNAME,
                        "CB_PASSWORD": CB_PASSWORD,
                        "CB_MCP_READ_ONLY_MODE": "true",
                    },
                ),
                timeout=60,
            ),
        )
    ],
)

print("Agent created successfully!")

Agent created successfully!


## Defining the Helper Function

We need a helper function to send questions to the agent and collect responses. Here's how it works:

1. **`Runner`** manages the agent's execution. It sends the user's message to the agent, which then decides what tools to call.
2. **`InMemorySessionService`** keeps track of the conversation history so the agent can reference previous messages.
3. The agent returns a stream of **events** — we collect only the final response text.

The `ask_agent` function below wraps all of this into a simple interface: pass in a question, get back the agent's answer.

In [6]:
import asyncio
from google.genai import errors as genai_errors

# Set up the session and runner (shared across all questions)
session_service = InMemorySessionService()
runner = Runner(
    agent=root_agent,
    app_name="couchbase_agent",
    session_service=session_service,
)


async def ask_agent(question: str, session_id: str) -> str:
    """Send a question to the agent. Retries on transient Gemini errors."""
    for attempt in range(4):
        try:
            response = ""
            async for event in runner.run_async(
                session_id=session_id,
                user_id="tutorial_user",
                new_message=types.Content(
                    role="user",
                    parts=[types.Part(text=question)],
                ),
            ):
                if event.is_final_response() and event.content and event.content.parts:
                    for part in event.content.parts:
                        if part.text:
                            response += part.text
            return response
        except (genai_errors.ClientError, genai_errors.ServerError):
            if attempt == 3:
                raise
            wait = 5 * (2 ** attempt)  # 5s, 10s, 20s
            print(f"  [Gemini API error, retrying in {wait}s...]")
            await asyncio.sleep(wait)
    return ""


print("Helper function ready!")

Helper function ready!


## Running the Agent

Now let's put our agent to work! We'll ask it a series of questions that demonstrate different capabilities:

1. **Database exploration** — Understanding what data is available
2. **SQL++ aggregation queries** — Complex queries with array operations
3. **Multi-step reasoning** — Finding flights AND hotels in one question
4. **Sightseeing recommendations** — Querying the landmarks collection
5. **Budget-based filtering** — Combining price filters with ratings

Each question triggers the agent to:
1. Analyze the question and decide which MCP tools to call
2. Call the appropriate tools (e.g., `get_scopes_and_collections_in_bucket`, `run_sql_plus_plus_query`)
3. Process the results from Couchbase
4. Generate a natural language answer

### Question 1: Tell me about the database

In [7]:
# Create a session for our conversation
session = await session_service.create_session(
    app_name="couchbase_agent", user_id="tutorial_user"
)

question = "Tell me about the database that you are connected to."
print(f"Question: {question}\n")
response = await ask_agent(question, session.id)
print(response)

Question: Tell me about the database that you are connected to.



I am connected to a Couchbase database. The following buckets are available: `beer-sample`, `gamesim-sample`, `travel-sample`, and `vector-search-testing`. I am specifically connected to the `travel-sample` bucket, and within that, the `inventory` scope. The collections available in the `inventory` scope are: `airline`, `airport`, `hotel`, `landmark`, and `route`.


### Question 2: Top hotels by rating

This question tests the agent's ability to write SQL++ queries with array operations. It needs to use `UNNEST` to flatten the reviews array and aggregate ratings — a common Couchbase pattern.

In [8]:
question = "List out the top 5 hotels by the highest aggregate rating."
print(f"Question: {question}\n")
response = await ask_agent(question, session.id)
print(response)

Question: List out the top 5 hotels by the highest aggregate rating.

The top 5 hotels by aggregate overall rating are:

1.  **Hotel Eldorado** - 109
2.  **Hafod Lon Holiday Apartment** - 70
3.  **Uist Outdoor Centre** - 53
4.  **Radisson Blu** - 50
5.  **Lochmaddy Hotel** - 50


### Question 3: Flight and hotel recommendations

This is a multi-step question that requires the agent to query two different collections (`route` for flights, `hotel` for accommodation) and combine the results into a single recommendation.

In [9]:
question = "Find flights from JFK to SFO and recommend a hotel in San Francisco under $200 a night."
print(f"Question: {question}\n")
response = await ask_agent(question, session.id)
print(response)

Question: Find flights from JFK to SFO and recommend a hotel in San Francisco under $200 a night.

Here are the flight options from JFK to SFO:

*   **Airlines:** B6, VX, AA, AS, US, UA, DL
*   **Source Airport:** JFK
*   **Destination Airport:** SFO
*   **Distance:** 4151.78 miles

Here are some hotels in San Francisco with prices under $200 that you might consider:

*   **USA Hostels San Francisco:** Private rooms: $64-$81
*   **Great Highway Inn:** $125–$155
*   **The Moffatt House:** $73+
*   **La Luna Inn:** $79-$129
*   **Travelodge Golden Gate:** $90 year round
*   **Inn on Castro:** $95–$190
*   **Country Hearth Inn:** $75-$110
*   **Laurel Inn:** $159-$209 (some rates within budget)
*   **Wharf Inn:** $160-$195
*   **Hostelling International-Fisherman's Wharf Hostel:** Private family rooms: $65-$100
*   **SW Hotel:** $140-$160
*   **Hotel Metropolis:** $99-$299 (some rates within budget)
*   **Hostelling International-Downtown:** Privates $69-$109
*   **San Remo Hotel:** $75-$

### Question 4: Sightseeing in the UK

This question tests the agent's ability to query the `landmark` collection and filter by country and activity type.

In [10]:
question = "I'm going to the UK for 1 week. Recommend some great spots to visit for sightseeing. Also mention the respective prices of those places for adults and kids."
print(f"Question: {question}\n")
response = await ask_agent(question, session.id)
print(response)

Question: I'm going to the UK for 1 week. Recommend some great spots to visit for sightseeing. Also mention the respective prices of those places for adults and kids.

Here are some great sightseeing spots in the United Kingdom, along with their prices for adults and children where available:

**London, England:**

*   **St Paul's Cathedral:** £16.50 for adults, £7.50 for children (6-17), £14.50 for concessions, £40 for a family ticket. Another entry in the database shows £9 for adults, £3.50 for children (7-16), and £8 for concessions, £21.50 for family, so prices may vary or there are different types of tours.
*   **Westminster Abbey (Deans Yd):** £12 for adults, £6 for concessions (seniors 60+, children 11-16, students with full-time student card), family ticket £18 (two adults and two children under 18), children under 11 free (maximum of two children per paying adult).
*   **Tower of London:** £14.50 for adults, £9.50 for children aged 5-16, £11 for concessions, £42 for a family (

### Question 5: Budget hotel search

This question demonstrates conversational context — since it follows the UK sightseeing question, the agent should understand the context and search for UK hotels within the budget.

In [11]:
question = "My budget is around 30 pounds a night. What will be the best hotel to stay in?"
print(f"Question: {question}\n")
response = await ask_agent(question, session.id)
print(response)

Question: My budget is around 30 pounds a night. What will be the best hotel to stay in?

Here are some hotels in the United Kingdom that appear to be within your budget of around £30 per night:

*   **Anis Louise Guest House** (Chesterfield): £27
*   **The Pines (B&B)** (Chippenham): Rooms from £25, includes full English Breakfast.
*   **Edale Youth Hostel** (Derbyshire): From £10 pppn (per person per night).
*   **Rhossili Bunkhouse** (Swansea): From £15 pppn.
*   **Moyle (Soerneog View Hostel)**: £15.00
*   **Lerwick Youth Hostel** (Lerwick): £17.50
*   **Pitlochry Backpackers Hotel** (Pitlochry): £17-£50 (some rates within your budget)
*   **Berwick Youth Hostel** (Berwick-upon-Tweed): Beds from £18
*   **Northumberland (Once Brewed YHA Hostel)**: Beds from £16
*   **Oban Backpackers** (Oban): from £16
*   **Highland (Glencoe Youth Hostel)**: £17.00
*   **Highland (Kinloch Hostel)**: £19.50 per person
*   **London (The Londonears Hostel)**: From £12 for a bed in a dormitory with up

## What Just Happened?

Behind the scenes, here's the flow for each question:

```
User Question
    ↓
Google ADK (LlmAgent)
    ↓ Gemini decides which tools to call
McpToolset → StdioConnectionParams → MCP Server subprocess
    ↓ Tool call (e.g., run_sql_plus_plus_query)
Couchbase MCP Server
    ↓ Executes SQL++ query against Couchbase
Couchbase Cluster (travel-sample bucket)
    ↓ Returns query results
MCP Server → McpToolset → LlmAgent
    ↓ Gemini formats results into natural language
Agent Response
```

The agent automatically:
1. Parsed your natural language question
2. Decided which Couchbase MCP tools to call
3. Generated correct SQL++ queries
4. Executed them against your live database
5. Formatted the results into a readable answer

## Next Steps

* **Try your own questions** — Modify the question strings above and re-run the cells
* **Enable write operations** — Set `CB_MCP_READ_ONLY_MODE` to `"false"` to allow document inserts, updates, and deletes
* **Use the ADK Web UI** — Run `adk web` from the command line to get an interactive chat interface
* **Explore the full tool list** — The Couchbase MCP Server provides 19+ tools for cluster management, schema discovery, querying, and performance analysis. See the [full documentation](https://github.com/Couchbase-Ecosystem/mcp-server-couchbase)
* **Check out the ADK integration page** — [Google ADK Couchbase Integration](https://google.github.io/adk-docs/integrations/couchbase/) for more configuration options

## Additional Resources

* [Couchbase MCP Server Repository](https://github.com/Couchbase-Ecosystem/mcp-server-couchbase)
* [Google ADK Documentation](https://google.github.io/adk-docs/)
* [Model Context Protocol](https://modelcontextprotocol.io/)
* [Couchbase Documentation](https://docs.couchbase.com/)
* [Couchbase Capella Free Tier](https://cloud.couchbase.com/)